# T-05 — Retrieval-Augmented Generation and the Study of Hallucination
## Phase 1 — Low-Code Baseline (YadYar Lite)

**Project:** YadYar Lite — Lightweight ML Learning Assistant  
**Topic:** T-05 — Retrieval-Augmented Generation and the Study of Hallucination  
**Phase:** Phase 1 only (Topic, Dataset, Baseline, Evaluation Plan, Analysis Plan)

---

## 0. Project Overview

This notebook builds a **lightweight Retrieval-Augmented Generation (RAG)**
pipeline on a small subset of **SQuAD 2.0** and prepares the ground for
studying **hallucination** behaviour in Phase 2.

The pipeline has three small parts:

1. **Retriever** — `sentence-transformers/all-MiniLM-L6-v2` embeds contexts; FAISS
   returns the top-k most similar contexts for a question.
2. **Generator** — `google/flan-t5-small` reads the question (optionally with a
   retrieved context) and writes a short answer.
3. **Two modes** — `generate_with_rag` (retriever + generator) and
   `generate_without_rag` (generator alone, the **No-RAG baseline**).

### What this notebook is

A small, reproducible, low-code baseline for T-05. The goal is to set up a
minimal RAG pipeline whose hallucination behaviour we can study in Phase 2 —
nothing more. We use only pretrained, off-the-shelf models; we do not train
or fine-tune anything.

### What this notebook is NOT

- It is **not** a product, an API, or a UI.
- It is **not** Phase 2 — full evaluation and error analysis are out of scope here.
- It does **not** train any model from scratch.
- It does **not** perform fine-tuning.
- It does **not** run full evaluation or final error analysis — that is **Phase 2**.

Phase 1 only requires: a narrowed question, a documented dataset, a reproducible
baseline, a short evaluation plan, and a simple analysis plan. This notebook
delivers exactly those, plus a tiny sanity check so we know the code runs.

## 1. Narrow Project Question

> **Can a lightweight RAG pipeline using dense retrieval reduce unsupported or
> hallucinated answers compared with a no-retrieval baseline on a small subset
> of SQuAD 2.0?**

**توضیح ساده (فارسی):**

آیا یک پایپ‌لاین RAG سبک که از dense retrieval (جستجوی معنایی با embedding) استفاده
می‌کند، می‌تواند تعداد پاسخ‌های بی‌پشتوانه یا hallucinated (ساخته‌شده توسط مدل) را
نسبت به حالت بدون بازیابی (No-RAG) روی یک زیرمجموعه کوچک از SQuAD 2.0 کاهش دهد؟

این سؤال محدود، قابل اجرا و قابل دفاع است زیرا:

- فقط یک مدل سبک آماده استفاده می‌شود (no training, no fine-tuning).
- فقط یک زیرمجموعه کوچک از یک دیتاست عمومی بررسی می‌شود.
- فقط دو حالت مقایسه می‌شود: **RAG** در برابر **No-RAG**.
- خروجی قابل اندازه‌گیری است: در فاز دوم با معیارهایی مثل Exact Match،
  Token F1 و Abstention Rate پاسخ‌ها ارزیابی می‌شوند.

این سؤال مستقیماً به الگوهای شکست ذکرشده در صورت پروژه برای T-05 اشاره می‌کند:
**absent-evidence hallucination** و **ignored-evidence hallucination**.

## 2. Dataset Description

| Field | Value |
|---|---|
| Dataset | **SQuAD 2.0** (Stanford Question Answering Dataset v2) |
| Source | Hugging Face Datasets: `rajpurkar/squad_v2` |
| Split used | `train` (a small subset of it) |
| Subset size | ~300 unique contexts, ~80 questions (mixed answerable / unanswerable) |

### Important fields

| Field | Meaning |
|---|---|
| `context` | The passage (paragraph) from which the answer should be extracted |
| `question` | The student-style question |
| `answers["text"]` | A list of gold answer strings (may be **empty** → unanswerable) |
| `answers["answer_start"]` | Character offsets of each gold answer in the context |

### ⚠️ Note on unanswerable questions

In the Hugging Face version of `squad_v2`, the `is_impossible` column is **not**
guaranteed to exist. The robust way to detect an unanswerable question is:

```python
is_unanswerable = len(example["answers"]["text"]) == 0
```

### Why SQuAD 2.0 fits T-05

- It is **public, small, well-documented, and widely used** for QA / RAG studies.
- It contains **both answerable and unanswerable** questions, which lets us study
  hallucination under two failure regimes: (a) the answer is in the context but the
  model misses it, and (b) the answer is **not** in the context, which is exactly
  where absent-evidence hallucination can appear.
- Each context is short (a paragraph), so embeddings and FLAN-T5-small both fit
  comfortably in a free Colab runtime.
- It is one of the suggested starting points for T-05 in the project catalogue.

## 3. Baseline Setup

### 3.1 Components

| Role | Tool / Model | Notes |
|---|---|---|
| Dataset loader | `datasets` (Hugging Face) | Loads `rajpurkar/squad_v2` |
| Embedding model | `sentence-transformers/all-MiniLM-L6-v2` | 384-dim, fast, CPU-friendly |
| Vector search | `faiss-cpu` (flat L2 index) | Simplest explainable ANN baseline |
| Generator (RAG) | `google/flan-t5-small` | ~80M params, runs on free Colab |
| No-RAG baseline | Same `flan-t5-small` **without** retrieved context | Apples-to-apples comparison |

### 3.2 Reproducibility

- Random seed is fixed (`SEED = 42`).
- Subset selection is deterministic given the seed.
- Models are pinned by their Hugging Face IDs.
- The whole pipeline runs in a free Colab CPU runtime in a few minutes.

### 3.3 Install dependencies

In [1]:
# Run once per Colab session. ~1–2 minutes on a fresh runtime.
!pip install -q datasets sentence-transformers faiss-cpu transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 85.2 MB/s eta 0:00:00:00:0100:01


### 3.4 Imports and seed

We import the four building blocks: dataset loading, embeddings, FAISS, and the
FLAN-T5 generator. A fixed seed makes the subset selection reproducible.

In [2]:
import os, random, numpy as np
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
import faiss
from transformers import T5ForConditionalGeneration, T5TokenizerFast

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Seed fixed:", SEED)

Seed fixed: 42


### 3.5 Load SQuAD 2.0

We load only the `train` split. The full split has ~130k examples, but we will
keep only a small subset in the next cell.

In [3]:
squad = load_dataset("rajpurkar/squad_v2", split="train")
print("Total SQuAD v2 train rows:", len(squad))
print("Sample row keys:", list(squad[0].keys()))
print("First context (truncated):", squad[0]["context"][:200])
print("First question:", squad[0]["question"])
print("First answers:", squad[0]["answers"])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


README.md:   0%|          | 0.00/8.92k [00:00<?, ?B/s]

squad_v2/train-00000-of-00001.parquet:   0%|          | 0.00/16.4M [00:00<?, ?B/s]

squad_v2/validation-00000-of-00001.parqu(…):   0%|          | 0.00/1.35M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/130319 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11873 [00:00<?, ? examples/s]

Total SQuAD v2 train rows: 130319
Sample row keys: ['id', 'title', 'context', 'question', 'answers']
First context (truncated): Beyoncé Giselle Knowles-Carter (/biːˈjɒnseɪ/ bee-YON-say) (born September 4, 1981) is an American singer, songwriter, record producer and actress. Born and raised in Houston, Texas, she performed in v
First question: When did Beyonce start becoming popular?
First answers: {'text': ['in the late 1990s'], 'answer_start': [269]}


### 3.6 Build a small, balanced subset

Goal: ~300 unique contexts and ~80 questions, with a mix of answerable and
unanswerable examples. We:

1. Shuffle the dataset deterministically.
2. Walk through it and collect **unique contexts** until we have ~300.
3. From the rows whose context is in that set, pick ~80 questions, making sure
   at least ~25% are unanswerable (empty `answers["text"]`) so Phase 2 can study
   absent-evidence hallucination.

In [4]:
TARGET_CONTEXTS = 300
TARGET_QUESTIONS = 80
TARGET_UNANSWERABLE = 20   # at least ~25% of the questions

# Deterministic shuffle
idxs = list(range(len(squad)))
random.Random(SEED).shuffle(idxs)

# Step 1: collect ~300 unique contexts
seen_contexts = {}          # context -> first row index
for i in idxs:
    ctx = squad[i]["context"]
    if ctx not in seen_contexts:
        seen_contexts[ctx] = i
    if len(seen_contexts) >= TARGET_CONTEXTS:
        break
contexts_list = list(seen_contexts.keys())
print("Unique contexts selected:", len(contexts_list))

# Step 2: gather candidate questions whose context is in our set
ctx_set = set(contexts_list)
cand_answerable, cand_unanswerable = [], []
for i in idxs:
    row = squad[i]
    if row["context"] not in ctx_set:
        continue
    is_unans = len(row["answers"]["text"]) == 0
    if is_unans:
        cand_unanswerable.append(i)
    else:
        cand_answerable.append(i)

# Step 3: pick a balanced subset
n_unans = min(TARGET_UNANSWERABLE, len(cand_unanswerable))
n_ans = TARGET_QUESTIONS - n_unans
selected = (cand_answerable[:n_ans] + cand_unanswerable[:n_unans])
random.Random(SEED).shuffle(selected)

subset = squad.select(selected)
print(f"Subset questions: {len(subset)} "
      f"(answerable={n_ans}, unanswerable={n_unans})")

Unique contexts selected: 300
Subset questions: 80 (answerable=60, unanswerable=20)


### 3.7 Embed contexts

We load `all-MiniLM-L6-v2` and embed the ~300 contexts. The model returns 384-dim
L2-normalised vectors, which work directly with FAISS.

In [5]:
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print("Embedding dim:", embedder.get_sentence_embedding_dimension())

context_embeddings = embedder.encode(
    contexts_list,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True,
)
print("Context embeddings shape:", context_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding dim: 384


/tmp/ipykernel_1083/1726465436.py:2: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dim:", embedder.get_sentence_embedding_dimension())


Batches:   0%|          | 0/10 [00:00<?, ?it/s]

Context embeddings shape: (300, 384)


### 3.8 Build FAISS index

We use a flat L2 index — the simplest possible vector search. There is no
approximation (no IVF, no PQ), so retrieval is exact and easy to explain in
the oral defence.

In [6]:
dim = context_embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(context_embeddings)
print("FAISS index size (vectors):", index.ntotal)

FAISS index size (vectors): 300


### 3.9 Retrieve function

Given a question, embed it, search the FAISS index, and return the top-k
context strings together with their indices.

In [7]:
def retrieve(question: str, k: int = 5):
    """Return top-k contexts for a question."""
    q_emb = embedder.encode(
        [question], convert_to_numpy=True, normalize_embeddings=True
    )
    distances, indices = index.search(q_emb, k)
    results = []
    for rank, (dist, idx) in enumerate(zip(distances[0], indices[0]), start=1):
        results.append({
            "rank": rank,
            "score": float(dist),
            "context_idx": int(idx),
            "context": contexts_list[idx],
        })
    return results

# Quick test on one question
sample_q = subset[0]["question"]
print("Question:", sample_q)
for r in retrieve(sample_q, k=3):
    print(f"  rank={r['rank']}  L2={r['score']:.4f}  ctx={r['context'][:80]}...")

Question: What was the name for a pub that could sell beer from more than one brewery?
  rank=1  L2=0.7893  ctx=After the development of the large London Porter breweries in the 18th century, ...
  rank=2  L2=1.4305  ctx=There are seven current masjids in the Greater Richmond area, with three more cu...
  rank=3  L2=1.4352  ctx="Whereas their Majesties have been Graciously Pleased to grant Letters patent to...


### 3.10 Load FLAN-T5-small generator

We use `google/flan-t5-small` (~80M params). It is small enough to run on a free
Colab CPU in seconds per question, and is instruction-tuned, so it follows short
prompts like "answer the question using the context".

In [8]:
MODEL_ID = "google/flan-t5-small"
tokenizer = T5TokenizerFast.from_pretrained(MODEL_ID)
generator = T5ForConditionalGeneration.from_pretrained(MODEL_ID)
print("Loaded:", MODEL_ID)

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.20k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.40k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loaded: google/flan-t5-small


### 3.11 Generation functions (RAG and No-RAG)

Both modes use the **same** generator so any difference in answer quality comes
from the retrieved context, not from a different model.

- `generate_with_rag(question)` — retrieves top-1 context and asks FLAN-T5 to
  answer using it. If the model thinks the context does not contain the answer,
  it is free to say "I don't know" (we do not force an answer).
- `generate_without_rag(question)` — passes only the question to FLAN-T5. This
  is the baseline that is most likely to hallucinate when the answer needs a
  specific fact from the corpus.

In [9]:
def _generate(prompt: str, max_new_tokens: int = 50) -> str:
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    out = generator.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,            # greedy for reproducibility
        num_beams=1,
    )
    return tokenizer.decode(out[0], skip_special_tokens=True).strip()

def generate_with_rag(question: str, k: int = 1):
    retrieved = retrieve(question, k=k)
    top_ctx = retrieved[0]["context"]
    prompt = (
        "Answer the question using the context. "
        "If the context does not contain the answer, say: I don't know.\n\n"
        f"Context: {top_ctx}\n\n"
        f"Question: {question}\n\n"
        "Answer:"
    )
    answer = _generate(prompt)
    return {"answer": answer, "retrieved_context": top_ctx,
            "all_retrieved": retrieved}

def generate_without_rag(question: str):
    prompt = f"Answer the question.\n\nQuestion: {question}\n\nAnswer:"
    answer = _generate(prompt)
    return {"answer": answer, "retrieved_context": None}

### 3.12 Sanity check (Phase 1 only)

We run **three** sample questions to confirm the pipeline works end-to-end:

1. One **answerable** question — compare RAG vs No-RAG.
2. One **unanswerable** question — see whether the RAG mode abstains or hallucinates.
3. One more **answerable** question for variety.

This is **not** evaluation — it is just a smoke test. Real metrics live in Phase 2.

In [10]:
def find_one(answerable: bool):
    for row in subset:
        is_unans = len(row["answers"]["text"]) == 0
        if answerable and not is_unans:
            return row
        if (not answerable) and is_unans:
            return row
    return None

samples = [find_one(answerable=True), find_one(answerable=False),
           find_one(answerable=True)]

for i, row in enumerate(samples, start=1):
    q = row["question"]
    gold = row["answers"]["text"]
    gold_str = gold[0] if gold else "(unanswerable — gold is empty)"
    rag = generate_with_rag(q)
    norag = generate_without_rag(q)
    print(f"\n===== Sample {i} =====")
    print("Question      :", q)
    print("Gold          :", gold_str)
    print("RAG context   :", rag["retrieved_context"][:140].replace('\n', ' '), "...")
    print("RAG answer    :", rag["answer"])
    print("No-RAG answer :", norag["answer"])


===== Sample 1 =====
Question      : What was the name for a pub that could sell beer from more than one brewery?
Gold          : a Free house
RAG context   : After the development of the large London Porter breweries in the 18th century, the trend grew for pubs to become tied houses which could on ...
RAG answer    : a Free house
No-RAG answer : st johns

===== Sample 2 =====
Question      : Nomadic hunter-gatherers are an exception to what rule?
Gold          : (unanswerable — gold is empty)
RAG context   : Hunter-gatherers tend to have an egalitarian social ethos, although settled hunter-gatherers (for example, those inhabiting the Northwest Co ...
RAG answer    : unanswerable
No-RAG answer : stagiation

===== Sample 3 =====
Question      : What was the name for a pub that could sell beer from more than one brewery?
Gold          : a Free house
RAG context   : After the development of the large London Porter breweries in the 18th century, the trend grew for pubs to become tied hous

## 4. Evaluation Plan (for Phase 2)

Phase 1 only requires a **plan**. We will compute the following metrics in Phase 2
on the same subset. Below each metric is a one-line explanation of what it measures.

### 4.1 Retrieval metrics

| Metric | What it measures |
|---|---|
| **Recall@5** | Of the gold context, how often is it inside the top-5 retrieved contexts? |
| **MRR** (Mean Reciprocal Rank) | How high is the gold context in the retrieved list, on average (1/rank)? |

### 4.2 Generation metrics

| Metric | What it measures |
|---|---|
| **Exact Match (EM)** | Fraction of answers that exactly match the gold (after normalisation) |
| **Token F1** | Token-level F1 between predicted and gold answer — partial credit |
| **ROUGE-L** | Longest common subsequence overlap between prediction and gold |

### 4.3 Hallucination-specific metric

| Metric | What it measures |
|---|---|
| **Abstention rate** | On unanswerable questions, the fraction where the model says "I don't know" instead of inventing an answer |
| **Hallucination rate** (manual, small sample) | On a sample of ~30 RAG answers, the fraction judged by a human as unsupported by the retrieved context |

### 4.4 Sanity check (already run above)

The 3-question smoke test in §3.12 confirms:

- the dataset loads and the subset is built;
- embeddings and the FAISS index work;
- both `generate_with_rag` and `generate_without_rag` produce a string;
- the No-RAG baseline sometimes invents facts when RAG abstains (or vice versa).

That is enough for Phase 1.

## 5. Simple Analysis Plan (for Phase 2)

We will group errors into **four categories** (the first three come directly from
the T-05 catalogue entry; the fourth is a data slice that is easy to compute on
SQuAD).

### 5.1 Error categories

| # | Error type | Definition |
|---|---|---|
| 1 | **Retrieval miss / absent-evidence hallucination** | The gold context was not in top-k, so the generator either hallucinates or abstains. |
| 2 | **Ignored-evidence hallucination** | The gold context *was* retrieved, but the generator still produced an answer unsupported by it. |
| 3 | **Unanswerable failure** | The question is unanswerable, but the model produced a confident (hallucinated) answer instead of abstaining. |
| 4 | **Short question vs long question** | Performance breakdown by question length (≤ 8 tokens vs > 8 tokens) — short questions may retrieve more weakly. |

### 5.2 Error-analysis template (Phase 2)

We will fill in the following small table for ~20 representative failures:

| Question | Gold answer | Retrieved context correct? | RAG answer | No-RAG answer | Error type | Explanation |
|---|---|---|---|---|---|---|
| _(filled in Phase 2)_ | | | | | | |

Each row will be short (one sentence per cell). The point is qualitative
understanding, not formal statistics.

## 6. Scope and Limitations

- This is **Phase 1 only**. We deliver a baseline and an evaluation plan, not
  final numbers.
- The subset is **very small** (~300 contexts, ~80 questions). Numbers from
  Phase 2 will be indicative, not statistically rigorous.
- The generator is **`flan-t5-small`** (~80M params). It is intentionally weak so
  that hallucination cases are visible and easy to discuss.
- The prompt is **a single, simple template**. No prompt engineering, no
  chain-of-thought, no self-consistency.
- Retrieval is **flat L2 with no reranker**. There is no approximate NN search
  and no hybrid (keyword + dense) retrieval.
- **Hallucination detection in Phase 1 is rule-based and preliminary** — we
  detect abstention with a substring rule (`"i don't know"` in the lower-cased
  answer). A proper human-judged hallucination rate is part of Phase 2.
- **Unanswerable detection** relies on `len(answers["text"]) == 0` because the
  Hugging Face version of `squad_v2` does not reliably expose `is_impossible`.